In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/advanced-dls-spring-2021/submission.csv
/kaggle/input/advanced-dls-spring-2021/train.csv
/kaggle/input/advanced-dls-spring-2021/test.csv


In [3]:
!ls

__notebook_source__.ipynb


In [9]:
sub = pd.read_csv('/kaggle/input/advanced-dls-spring-2021/submission.csv')
test = pd.read_csv('/kaggle/input/advanced-dls-spring-2021/test.csv')
data = pd.read_csv('/kaggle/input/advanced-dls-spring-2021/train.csv')

In [10]:
sub

,Id,Churn
0,0,0.5
1,1,0.5
2,2,0.5
3,3,0.5
4,4,0.5
...,...,...
1756,1756,0.5
1757,1757,0.5
1758,1758,0.5
1759,1759,0.5


In [11]:
# Для вашего удобства списки с именами разных колонок

# Числовые признаки
num_cols = [
    'ClientPeriod',
    'MonthlySpending',
    'TotalSpent'
]

# Категориальные признаки
cat_cols = [
    'Sex',
    'IsSeniorCitizen',
    'HasPartner',
    'HasChild',
    'HasPhoneService',
    'HasMultiplePhoneNumbers',
    'HasInternetService',
    'HasOnlineSecurityService',
    'HasOnlineBackup',
    'HasDeviceProtection',
    'HasTechSupportAccess',
    'HasOnlineTV',
    'HasMovieSubscription',
    'HasContractPhone',
    'IsBillingPaperless',
    'PaymentMethod'
]

feature_cols = num_cols + cat_cols
target_col = 'Churn'

In [7]:
!pip install catboost

In [12]:
from sklearn.model_selection import train_test_split
X = data.drop(['Churn'], axis=1)
Y = data['Churn']

In [13]:
x,xv,y,yv = train_test_split(X,Y,test_size=0.1,stratify=Y)

In [19]:
from catboost import CatBoostClassifier
model = CatBoostClassifier(
    iterations=100000,
    learning_rate=0.001,
    loss_function='Logloss',
    eval_metric='AUC',
    early_stopping_rounds=2000,
 
)

model.fit(
    x, y,
    eval_set=(xv, yv),
    cat_features=cat_cols,
    verbose=False,

)

In [20]:
from sklearn.metrics import roc_auc_score,f1_score
pred = model.predict(xv)
pred_prob = model.predict_proba(xv)
# f1_score(yv,pred)
roc_auc_score(yv,pred_prob[:,1])

0.8365615200147574

In [22]:
test_proba = model.predict_proba(test)
test_df = pd.DataFrame()
test_df['Id'] = np.array([i for i in range(len(test_proba))])
test_df['Churn'] = test_proba[:,1]
test_df.to_csv("/kaggle/working/submission.csv", index = False)